# ML/AI Algorithms - Pediatric Nephrology Platform

Ce notebook implémente les mêmes algorithmes ML que le backend Spring Boot,
en Python avec visualisations pour mieux comprendre leur fonctionnement.

## Modules couverts
- **Forum** : Clustering, Classification, Prédiction, Recommandation
- **Kidney Transplant** : Clustering, Classification, Prédiction, Recommandation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import math
import random
from sklearn.datasets import make_blobs
from scipy import spatial

plt.style.use('ggplot')
%matplotlib inline
print('OK - imports ready')

---
## 1. PARTIE COMMUNE : Utilitaires ML

In [ ]:
STOP_WORDS = set('''le la les de du des un une et est sont dans pour sur avec par pas plus moins tres
a ont etre avoir faire je tu il elle nous vous ils elles ce cette ces mon ton son mes tes ses
notre votre leur nos vos leurs the a an and or but in on at to for of by with from is are
was were be been being have has had do does did will would could should may might shall
can need dare this that these those i you he she it we they me him her us them my your his its our their'''.split())

def tokenize(text):
    """Tokenisation + nettoyage + stop words"""
    if not text: return []
    words = text.lower().split()
    return [w.strip('.,!?;:()[]{}') for w in words if len(w) > 2 and w not in STOP_WORDS]

def tf(tokens):
    """Term Frequency"""
    cnt = Counter(tokens)
    total = len(tokens)
    return {w: c/total for w, c in cnt.items()}

# Demo
sample = "Mon enfant a subi une greffe renale et nous cherchons du soutien"
print('Tokens:', tokenize(sample))
print('TF:', tf(tokenize(sample)))

In [ ]:
class TFIDFCalculator:
    def __init__(self):
        self.vocab = []
        self.idf = {}
        self.doc_vectors = []

    def fit(self, tokenized_docs):
        df = Counter()
        for doc in tokenized_docs:
            df.update(set(doc))
        n = len(tokenized_docs)
        self.vocab = list(df.keys())
        self.idf = {w: math.log((n + 1) / (c + 1)) + 1 for w, c in df.items()}
        self.doc_vectors = [self._transform(doc) for doc in tokenized_docs]
        return self

    def _transform(self, tokens):
        vec = np.zeros(len(self.vocab))
        tf_vals = tf(tokens)
        for i, w in enumerate(self.vocab):
            vec[i] = tf_vals.get(w, 0) * self.idf.get(w, 1)
        return vec

    def transform(self, tokens):
        return self._transform(tokens)

# Demo TF-IDF
docs = [
    "ma greffe renale est un succes",
    "rejet aigu apres transplantation",
    "soutien pour les parents apres greffe",
    "creatinine elevee risque de rejet"
]
tokenized = [tokenize(d) for d in docs]
tfidf = TFIDFCalculator().fit(tokenized)
print('Vocabulaire:', tfidf.vocab)
print('IDF:', {w: round(v, 2) for w, v in tfidf.idf.items()})
print('Matrice TF-IDF shape:', tfidf.doc_vectors[0].shape)

In [ ]:
def cosine_similarity(a, b):
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return dot / norm if norm > 0 else 0

def top_k_similarities(query_vec, doc_vectors, k=3):
    sims = [(i, cosine_similarity(query_vec, v)) for i, v in enumerate(doc_vectors)]
    sims.sort(key=lambda x: -x[1])
    return sims[:k]

# Demo : similarité entre documents
for i, v1 in enumerate(tfidf.doc_vectors):
    for j, v2 in enumerate(tfidf.doc_vectors):
        if i < j:
            sim = cosine_similarity(v1, v2)
            print(f"Doc {i} <-> Doc {j}: {sim:.3f}")

---
## 2. K-MEANS CLUSTERING

Regroupe les données en `k` clusters basés sur la distance euclidienne.

In [ ]:
class KMeansClustering:
    def __init__(self, k, max_iter=100):
        self.k = k
        self.max_iter = max_iter
        self.centroids = None
        self.clusters = None

    def fit(self, data):
        n, dim = data.shape
        random.seed(42)
        idx = random.sample(range(n), min(self.k, n))
        self.centroids = data[idx].copy()

        for _ in range(self.max_iter):
            self.clusters = {i: [] for i in range(self.k)}
            for i, point in enumerate(data):
                nearest = np.argmin([np.linalg.norm(point - c) for c in self.centroids])
                self.clusters[nearest].append(i)

            new_centroids = np.zeros_like(self.centroids)
            converged = True
            for i in range(self.k):
                if self.clusters[i]:
                    new_centroids[i] = data[self.clusters[i]].mean(axis=0)
                else:
                    new_centroids[i] = self.centroids[i]
                if np.linalg.norm(new_centroids[i] - self.centroids[i]) > 1e-6:
                    converged = False
            self.centroids = new_centroids
            if converged:
                break
        return self

    def predict(self, point):
        return np.argmin([np.linalg.norm(point - c) for c in self.centroids])

# Générer des données de test (comme les features de greffe)
X, y_true = make_blobs(n_samples=50, centers=3, n_features=2, random_state=42)
kmeans = KMeansClustering(k=3).fit(X)

# Visualisation
colors = ['red', 'blue', 'green']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for i in range(3):
    pts = X[kmeans.clusters[i]]
    ax1.scatter(pts[:, 0], pts[:, 1], c=colors[i], label=f'Cluster {i}', alpha=0.6)
ax1.scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1], c='black', marker='X', s=200, label='Centroïdes')
ax1.set_title('K-Means Clustering (données simulées)')
ax1.legend()

ax2.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.6)
ax2.set_title('Vrais clusters')
plt.tight_layout()
plt.show()

# Application Forum: cluster posts par thème
forum_posts = [
    "comment gérer les effets secondaires du tacrolimus",
    "expérience avec la dialyse péritonéale",
    "témoignage greffe rénale après 5 ans",
    "traitement immunosuppresseur et vaccination",
    "mon enfant a besoin d'un donneur compatible",
    "résultats de créatinine après transplantation"
]
tokenized_posts = [tokenize(p) for p in forum_posts]
tfidf_forum = TFIDFCalculator().fit(tokenized_posts)
mat = np.array(tfidf_forum.doc_vectors)
kmeans_forum = KMeansClustering(k=2).fit(mat)
print("\n=== Clustering Forum ===")
for cluster_id, indices in kmeans_forum.clusters.items():
    print(f"\nCluster {cluster_id}:")
    for idx in indices:
        print(f"  - {forum_posts[idx]}")

---
## 3. NAIVE BAYES CLASSIFICATION

Classification probabiliste basée sur le théorème de Bayes.

In [ ]:
class NaiveBayesClassifier:
    def __init__(self):
        self.classes = []
        self.priors = {}
        self.word_probs = {}
        self.vocab = set()
        self.total_docs = 0

    def fit(self, labeled_docs):
        self.classes = list(labeled_docs.keys())
        word_counts = {c: Counter() for c in self.classes}
        doc_lengths = {c: 0 for c in self.classes}

        for cls, tokens in labeled_docs.items():
            word_counts[cls].update(tokens)
            doc_lengths[cls] = len(tokens)
            self.vocab.update(tokens)
            self.total_docs += len(tokens)

        for cls in self.classes:
            self.priors[cls] = doc_lengths[cls] / self.total_docs
            total_words = sum(word_counts[cls].values())
            vocab_size = len(self.vocab)
            self.word_probs[cls] = {
                w: (word_counts[cls].get(w, 0) + 1) / (total_words + vocab_size)
                for w in self.vocab
            }
        return self

    def predict(self, tokens):
        scores = {}
        for cls in self.classes:
            log_prob = math.log(self.priors[cls])
            for token in tokens:
                p = self.word_probs[cls].get(token, 1 / (len(self.vocab) + 1))
                log_prob += math.log(p)
            scores[cls] = log_prob
        return scores

    def predict_class(self, tokens):
        return max(self.predict(tokens).items(), key=lambda x: x[1])[0]

    def confidence(self, tokens):
        scores = self.predict(tokens)
        exp_scores = {k: math.exp(v) for k, v in scores.items()}
        total = sum(exp_scores.values())
        return max(exp_scores.values()) / total if total > 0 else 0

# Exemple : Classification de posts Forum
forum_data = {
    'MEDICAL_QUESTION': tokenize("quel traitement pour insuffisance renale chronique comment diagnostiquer"),
    'EXPERIENCE_SHARING': tokenize("mon experience avec dialyse temoignage apres greffe j ai vecu"),
    'SUPPORT': tokenize("besoin aide soutien encouragement pour mon enfant malade"),
    'INFORMATION': tokenize("nouvelle etude sur immunosuppresseur actualite transplantation")
}
nb = NaiveBayesClassifier().fit(forum_data)

test_posts = [
    "comment faire pour reduire la creatinine",
    "mon histoire de greffe renale"]

print('=== Classification de Posts Forum ===\n')
for post in test_posts:
    tokens = tokenize(post)
    cls = nb.predict_class(tokens)
    conf = nb.confidence(tokens)
    scores = nb.predict(tokens)
    exp_scores = {k: math.exp(v) for k, v in scores.items()}
    sorted_scores = sorted(exp_scores.items(), key=lambda x: -x[1])
    scores_str = ', '.join(f'{k}={v:.2f}' for k, v in sorted_scores)
    print(f'Post: {post}')
    print(f'  Predicted: {cls} (confidence: {conf:.1%})')
    print(f'  Scores: {scores_str}\n')

# Visualisation des probabilités
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, post in enumerate(test_posts):
    tokens = tokenize(post)
    scores = nb.predict(tokens)
    classes = list(scores.keys())
    probs = [math.exp(scores[c]) for c in classes]
    ax[i].bar(classes, probs, color=['#2196F3', '#FF9800', '#4CAF50', '#E91E63'])
    ax[i].set_title(f"'{post[:30]}...'")
    ax[i].tick_params(axis='x', rotation=15)
    ax[i].set_ylabel('Probabilité')
plt.tight_layout()
plt.show()

---
## 4. RÉGRESSION LINÉAIRE

Prédiction de valeurs continues (popularité, survie du greffon...).

In [ ]:
def linear_regression(x, y):
    """Régression linéaire simple: y = ax + b"""
    n = len(x)
    sx = sum(x)
    sy = sum(y)
    sxy = sum(x[i] * y[i] for i in range(n))
    sx2 = sum(xi**2 for xi in x)
    a = (n * sxy - sx * sy) / (n * sx2 - sx**2) if (n * sx2 - sx**2) != 0 else 0
    b = (sy - a * sx) / n
    return a, b

def r_squared(x, y, a, b):
    y_pred = [a * xi + b for xi in x]
    y_mean = np.mean(y)
    ss_res = sum((y[i] - y_pred[i])**2 for i in range(len(y)))
    ss_tot = sum((yi - y_mean)**2 for yi in y)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0

# Exemple : Prédiction de popularité des posts
# Features: âge du post en heures, Target: nombre de vues
hours = np.array([1, 2, 5, 10, 24, 48, 72, 120, 168, 336])
views = np.array([5, 8, 15, 28, 45, 62, 78, 95, 110, 130])

a, b = linear_regression(hours, views)
r2 = r_squared(hours, views, a, b)
print(f"Régression: vues = {a:.2f} * heures + {b:.2f}")
print(f"R² = {r2:.3f}")

# Prédiction à 7 jours (168h)
pred_7d = a * 168 + b
print(f"Prédiction vues à 7 jours: {pred_7d:.0f}")

# Visualisation
plt.figure(figsize=(10, 5))
plt.scatter(hours, views, color='blue', s=80, label='Données réelles')
x_line = np.linspace(0, 400, 100)
plt.plot(x_line, a * x_line + b, 'r-', label=f'Tendance (R²={r2:.2f})')
plt.scatter([168], [pred_7d], color='green', s=150, marker='*', label=f'Prédiction 7j: {pred_7d:.0f}')
plt.xlabel('Heures depuis publication')
plt.ylabel('Nombre de vues')
plt.title('Prédiction de popularité des posts Forum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. APPLICATION : Prédiction de Survie du Greffon

In [ ]:
def estimate_graft_survival(transplant_type, cold_ischemia, warm_ischemia,
                             delayed_function, acute_rejection, graft_failure,
                             primary_function, infection, peak_creatinine):
    """Estimation de la survie du greffon en mois (identique au backend)"""
    base = 60  # 5 ans de base
    if transplant_type == 'LIVING_DONOR': base += 24
    if transplant_type == 'DECEASED_DONOR': base -= 12
    base -= cold_ischemia * 0.5
    base -= warm_ischemia * 0.3
    if delayed_function: base -= 18
    if acute_rejection: base -= 24
    if graft_failure: base = 6
    if primary_function: base += 12
    if infection: base -= 6
    if peak_creatinine > 3: base -= 12
    return max(1, base)

# Scénarios de test
scenarios = [
    {'name': 'Donneur vivant, faible ischémie', 'type': 'LIVING_DONOR', 'ci': 2, 'wi': 30,
     'df': False, 'ar': False, 'gf': False, 'pf': True, 'inf': False, 'pc': 1.2},
    {'name': 'Donneur décédé, ischémie longue', 'type': 'DECEASED_DONOR', 'ci': 28, 'wi': 60,
     'df': True, 'ar': True, 'gf': False, 'pf': False, 'inf': True, 'pc': 3.5},
    {'name': 'Donneur vivant, rejet aigu', 'type': 'LIVING_DONOR', 'ci': 4, 'wi': 40,
     'df': False, 'ar': True, 'gf': False, 'pf': True, 'inf': False, 'pc': 2.8},
]

print("=== Prédiction de Survie du Greffon ===\n")
for s in scenarios:
    surv = estimate_graft_survival(s['type'], s['ci'], s['wi'], s['df'], s['ar'],
                                    s['gf'], s['pf'], s['inf'], s['pc'])
    print(f"{s['name']}:")
    print(f"  Survie estimée: {surv:.0f} mois ({surv/12:.1f} ans)")
    if surv > 60:
        print(f"  Pronostic: EXCELLENT")
    elif surv > 24:
        print(f"  Pronostic: BON")
    else:
        print(f"  Pronostic: RÉSERVÉ")
    print()

# Visualisation comparative
names = [s['name'][:20] for s in scenarios]
survivals = [estimate_graft_survival(s['type'], s['ci'], s['wi'], s['df'], s['ar'],
                                     s['gf'], s['pf'], s['inf'], s['pc']) for s in scenarios]
colors_surv = ['green' if s > 60 else 'orange' if s > 24 else 'red' for s in survivals]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, survivals, color=colors_surv, alpha=0.7)
plt.axhline(y=60, color='green', linestyle='--', label='Seuil excellent (5 ans)')
plt.axhline(y=24, color='orange', linestyle='--', label='Seuil bon (2 ans)')
plt.ylabel('Survie estimée (mois)')
plt.title('Prédiction de survie du greffon par scénario')
plt.legend()
plt.xticks(rotation=15)
for bar, surv in zip(bars, survivals):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{surv:.0f}m', ha='center', va='bottom')
plt.tight_layout()
plt.show()

---
## 6. APPLICATION : Clustering de Greffes Rénales

Regroupement par profil de risque basé sur des features cliniques.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Simuler des données de greffes (comme dans le backend)
np.random.seed(42)
n_transplants = 40

transplant_data = {
    'cold_ischemia': np.random.normal(15, 8, n_transplants),      # 0-40 min
    'warm_ischemia': np.random.normal(45, 15, n_transplants),     # 20-90 min
    'blood_loss': np.random.normal(400, 200, n_transplants),      # 50-1200 ml
    'hospital_stay': np.random.normal(12, 5, n_transplants),      # 3-30 jours
    'peak_creatinine': np.random.normal(2.0, 1.0, n_transplants), # 0.5-6.0
    'graft_survival': np.random.normal(48, 24, n_transplants)     # 1-120 mois
}

# Créer quelques outliers (échecs, rejets)
for i in range(5):
    idx = np.random.randint(0, n_transplants)
    transplant_data['cold_ischemia'][idx] = np.random.uniform(30, 40)
    transplant_data['graft_survival'][idx] = np.random.uniform(1, 6)
    transplant_data['peak_creatinine'][idx] = np.random.uniform(4, 7)

X = np.column_stack([v for v in transplant_data.values()])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Means
kmeans_tx = KMeansClustering(k=3).fit(X_scaled)

# Visualisation en 2D (PCA simplifiée: 2 premières features)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_tx = ['red', 'blue', 'green']
for i in range(3):
    pts = X_scaled[kmeans_tx.clusters[i]]
    axes[0].scatter(pts[:, 0], pts[:, 1], c=colors_tx[i], label=f'Cluster {i}', alpha=0.7, s=80)
axes[0].scatter(kmeans_tx.centroids[:, 0], kmeans_tx.centroids[:, 1],
                c='black', marker='X', s=200, label='Centroïdes')
axes[0].set_xlabel('Ischémie froide (normalisée)')
axes[0].set_ylabel('Ischémie chaude (normalisée)')
axes[0].set_title('Clustering des greffes par profil de risque')
axes[0].legend()

# Analyse des clusters
cluster_labels = []
for i in range(3):
    indices = kmeans_tx.clusters[i]
    avg_survival = np.mean(transplant_data['graft_survival'][indices])
    avg_ischemia = np.mean(transplant_data['cold_ischemia'][indices])
    if avg_survival < 12:
        label = 'HAUT RISQUE'
    elif avg_survival < 36:
        label = 'RISQUE MODÉRÉ'
    else:
        label = 'FAIBLE RISQUE'
    cluster_labels.append(label)
    axes[1].bar(i, len(indices), color=colors_tx[i], alpha=0.7)
    axes[1].text(i, len(indices) + 0.5, f'{label}\n{len(indices)} greffes',
                ha='center', va='bottom')
axes[1].set_xticks(range(3))
axes[1].set_xticklabels([f'Cluster {i}' for i in range(3)])
axes[1].set_ylabel('Nombre de greffes')
axes[1].set_title('Distribution des clusters')

plt.tight_layout()
plt.show()

print("\n=== Analyse des Clusters ===")
for i in range(3):
    indices = kmeans_tx.clusters[i]
    print(f"\nCluster {i} - {cluster_labels[i]} ({len(indices)} greffes):")
    print(f"   Survie moyenne: {np.mean(transplant_data['graft_survival'][indices]):.1f} mois")
    print(f"   Ischémie froide: {np.mean(transplant_data['cold_ischemia'][indices]):.1f} min")

---
## 7. SYSTÈME DE RECOMMANDATION

Recommandation de posts similaires basée sur la similarité cosinus TF-IDF.

In [ ]:
class ForumRecommendationEngine:
    def __init__(self, posts):
        self.posts = posts
        tokenized = [tokenize(p) for p in posts]
        self.tfidf = TFIDFCalculator().fit(tokenized)
        self.vectors = np.array(self.tfidf.doc_vectors)

    def recommend_for_user(self, user_interests, top_n=3):
        user_vec = self.tfidf.transform(tokenize(user_interests))
        sims = [(i, cosine_similarity(user_vec, v)) for i, v in enumerate(self.vectors)]
        sims.sort(key=lambda x: -x[1])
        return [(self.posts[i], score) for i, score in sims[:top_n]]

    def similar_to(self, post_idx, top_n=3):
        query = self.vectors[post_idx]
        sims = [(i, cosine_similarity(query, v)) for i, v in enumerate(self.vectors)
                if i != post_idx]
        sims.sort(key=lambda x: -x[1])
        return [(self.posts[i], score) for i, score in sims[:top_n]]

# Base de posts Forum
forum_posts = [
    "Comment gérer les effets secondaires du tacrolimus après greffe",
    "Expérience avec la dialyse péritonéale chez mon enfant de 5 ans",
    "Témoignage: 10 ans après ma greffe rénale, tout va bien",
    "Question: quel régime alimentaire après transplantation rénale?",
    "Soutien aux parents d'enfants dialysés, vous n'êtes pas seuls",
    "Information sur les nouveaux immunosuppresseurs en 2026",
    "Mon histoire: de l'insuffisance rénale à la greffe",
    "Résultats de créatinine inquiétants après transplantation"
]

engine = ForumRecommendationEngine(forum_posts)

print("=== Système de Recommandation Forum ===\n")

# Recommandation pour un utilisateur intéressé par les traitements
user_query = "effets secondaires traitement immunosuppresseur médicaments"
print(f"🔍 Profil utilisateur: '{user_query}'")
print("Recommandations:")
for post, score in engine.recommend_for_user(user_query, 3):
    print(f"  • {post} (score: {score:.2f})")

print("\n---\n")

# Posts similaires au post #2 (témoignage)
print(f"🔍 Posts similaires à: '{forum_posts[2]}'")
for post, score in engine.similar_to(2, 3):
    print(f"  • {post} (score: {score:.2f})")

# Visualisation des similarités
sim_matrix = np.array([[cosine_similarity(engine.vectors[i], engine.vectors[j])
                        for j in range(len(forum_posts))]
                       for i in range(len(forum_posts))])

plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(label='Similarité')
plt.title('Matrice de similarité entre posts')
plt.xlabel('Post #')
plt.ylabel('Post #')
for i in range(len(forum_posts)):
    for j in range(len(forum_posts)):
        plt.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()

---
## 8. RÉSUMÉ DES ALGORITHMES

| Algorithme | Type | Module | Usage |
|---|---|---|---|
| **TF-IDF** | Feature extraction | Forum | Vectorisation du texte des posts |
| **K-Means** | Clustering | Forum + Transplant | Regroupement posts par thème / greffes par risque |
| **Naive Bayes** | Classification | Forum + Transplant | Catégorisation posts / prédiction outcome greffe |
| **Régression Linéaire** | Prédiction | Forum + Transplant | Popularité posts / survie greffon |
| **Similarité Cosinus** | Recommandation | Forum + Transplant | Posts similaires / greffes similaires |

Tous ces algorithmes sont implémentés:
- **Backend Java** : `Back/src/main/java/.../ml/common/` (classes pures, zéro dépendance ML externe)
- **API REST** : `/api/forum/ai/*` et `/api/kidney-transplants/ai/*`
- **Frontend Angular** : `Front/src/app/ml/` (dashboard, clustering, classification, prediction, recommendation)
- **Ce notebook Python** : `notebooks/ml_algos_demo.ipynb`